# Feature Engineering 

**Project:** Airbnb Pricing Analysis

**Team:** Jessie Dicker, Norma Arredondo

**Date:** April 4, 2026




## Table of Contents
1. Load Preprocessed Data
2. Create New Features
3. Encode Categorical Features
4. Scale Numerical Features
5. Feature Selection
6. Save Final Data
7. Key Findings Summary

In [25]:
# import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# settings
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)

# 1. Load Preprocessed Data

This section loads the preprocessed train and test splits from Notebook 2 and confirms the data.

In [5]:
# load preprocessed data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')
print(f'X_train Loaded: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns')
print(f'y_train Loaded: {y_train.shape[0]:,} rows x {y_train.shape[1]} columns')

print(f'X_test Loaded: {X_test.shape[0]:,} rows x {X_test.shape[1]} columns')
print(f'y_test Loaded: {y_test.shape[0]:,} rows x {y_test.shape[1]} columns')

X_train Loaded: 39,107 rows x 10 columns
y_train Loaded: 39,107 rows x 1 columns
X_test Loaded: 9,777 rows x 10 columns
y_test Loaded: 9,777 rows x 1 columns


In [6]:
X_train.head()

,neighbourhood_group,neighbourhood,latitude,longitude,room_type,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,Brooklyn,Williamsburg,40.71151,-73.95937,Entire home/apt,1,1,1.00,2.0,1
1,Brooklyn,Williamsburg,40.71106,-73.94884,Entire home/apt,5,15,0.97,1.0,83
2,Manhattan,Inwood,40.86271,-73.92872,Entire home/apt,5,39,1.28,1.0,12
3,Brooklyn,Bushwick,40.70364,-73.92679,Private room,2,0,0.00,1.0,57
4,Brooklyn,Bedford-Stuyvesant,40.69360,-73.94653,Private room,1,1,0.02,1.0,0


In [7]:
# check column types

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

print(f'Numerical ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical ({len(categorical_cols)}): {categorical_cols}')

Numerical (7): ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
Categorical (3): ['neighbourhood_group', 'neighbourhood', 'room_type']


### Load Preprocessed Data Observation

**Shapes of X_train and X_test**

X_train: (39107, 10)

X_test: (9777, 10)

**Number of Numerical and Categorical Features**

There are 7 Numerical Features: 

`latitude`, `longitude`, `minimum_nights`, `number_of_reviews`, `reviews_per_month`, `calculated_host_listings_count`, and `availability_365`

There are 3 Categorical Features: 

`neighbourhood_group`, `neighbourhood`, and `room_type`

**The columns we planned to engineer new features from:**

- `latitude` & `longitude` — as a location-based feature
- `minimum_nights` — can be a log transformation
- `number_of_reviews` — can be binary flagged
- `number_of_reviews` & `availability_365` — can be combined into 
  a ratio feature


# 2. Create New Features

This section engineers new features using different techniques.

### 2.1 Log Transformation 

In [8]:
# Log transform minimum_nights to reduce right skew
X_train['log_minimum_nights'] = np.log1p(X_train['minimum_nights'])
X_test['log_minimum_nights'] = np.log1p(X_test['minimum_nights']) # identical as train

# no fitting needed
print("Original minimum_nights skew:", X_train['minimum_nights'].skew().round(2))
print("Log minimum_nights skew:     ", X_train['log_minimum_nights'].skew().round(2))

Original minimum_nights skew: 1.29
Log minimum_nights skew:      0.6


**Created Feature:**

Applied Log Transformation on `minimum_nights` to create `log_minimum_nights`. log(1 + x) was used to compute.


**Reason it was created:**

From our Exploratory Data Analysis, the `minimum_nights` is heavily right-skewed (skew = 1.29). The long tail can disrupt the model, so reducing the skew can help distribute it evenly. 


**Skewed Value:**

Original minimum_nights skew: 1.29
Log minimum_nights skew:      0.6

### 2.2 Location-based Feature

In [9]:
# Times Square: 40.7580° N, 73.9855° W
# Using longitude and latitude 

# Haversine distance to Times Square
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two lat/lon points."""
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Distance to Times Square
X_train['dist_to_times_square'] = haversine_distance(
    X_train['latitude'], X_train['longitude'], 40.7580, -73.9855)
X_test['dist_to_times_square'] = haversine_distance(
    X_test['latitude'], X_test['longitude'], 40.7580, -73.9855)

# no fitting needed
print("Distance feature created!")
print(f"Avg distance to Times Square: {X_train['dist_to_times_square'].mean():.1f} km")

Distance feature created!
Avg distance to Times Square: 7.1 km


**Created Feature:**

`dist_to_times_square` was created using the Location-Based technique by calculating the kilometers using the Haversine distance formula with`longitude` and `latitude`.


**Reason it was created:**

In the Exploratory Data Analysis, `longitude` was the strongest predictor of price (r = -0.15). Geographically matters in this data, and Times Square is a popular landmark in New York City, which makes a good central landmark.  


**Average Distance:**

Average  distance to Times Square: 7.1 km

### 2.3 Binning 

In [10]:
# Binned Distance (dist_to_times_square)
bin_edges = [0, 2, 5, 10, 50]
bin_labels = ['Very Central', 'Central', 'Outer', 'Remote']

X_train['location_tier'] = pd.cut(
    X_train['dist_to_times_square'],
    bins=bin_edges,
    labels=bin_labels
)
X_test['location_tier'] = pd.cut(
    X_test['dist_to_times_square'],
    bins=bin_edges,
    labels=bin_labels
)

print("Training distribution:")
print(X_train['location_tier'].value_counts().sort_index())

Training distribution:
location_tier
Very Central     4466
Central          9152
Outer           17494
Remote           7995
Name: count, dtype: int64


**Created Feature:**

Binned the `dist_to_times_square` (the new feature) column into four distance zones using pd.cut(), to create `location_tier`.

- Very Central: 0-2 km
- Central: 2-5 km
- Outer: 5-10 km
- Remote: 10+ km


**Reason it was created:**

Instead of treating every kilometer as equal, binning captures the price impact of location. 1 km to Times Square isn't the same as 3km to Times Square.


**Training Distribution:**

- Very Central: 4,466 listings
- Central: 9,152 listings
- Outer: 17,494 listings
- Remote: 7,995 listings

### 2.4 Binary Flag

In [11]:
# Using number_of_reviews
# Flag listings with zero reviews
X_train['is_new_listing'] = (X_train['number_of_reviews'] == 0).astype(int)
X_test['is_new_listing'] = (X_test['number_of_reviews'] == 0).astype(int)

print("New listing flag distribution (train):")
print(X_train['is_new_listing'].value_counts())
print(f"\n{X_train['is_new_listing'].mean()*100:.1f}% of listings have no reviews")

New listing flag distribution (train):
is_new_listing
0    31086
1     8021
Name: count, dtype: int64

20.5% of listings have no reviews


**Created Feature:**

Created a binary column, `is_new_listing`, that flags listings where the listing has never received a review.

1 = listing has no reviews
0 = listing has at least one review


**Reason it was created:**

A Listing with zero reviews could mean a new listing, or it hasn't been booked yet. Listing with no reviews is a meaningful pricing signal that the raw `number_of_reviews` column could not be separated on its own. 

In the Data preprocessing notebook, we imputed missing values in `reviews_per_month` with no reviews. This flag is consistent with that earlier decision.


**Training Distribution:**

1 (no reviews): 31,086 listings

0 (at least one review): 8,021 listings

### 2.5 Ratio Feature

In [12]:
# Ratio using number_of_reviews and availability_365
X_train['review_density'] = (
    X_train['number_of_reviews'] / (X_train['availability_365'] + 1) # + 1 to avoid division by zero
)
X_test['review_density'] = (
    X_test['number_of_reviews'] / (X_test['availability_365'] + 1)  # + 1 to avoid division by zero
)

# Fit median on training data only
review_density_median = X_train['review_density'].median()

# Apply to both sets
X_train['review_density'] = X_train['review_density'].replace(
    [np.inf, -np.inf], review_density_median
)
X_test['review_density'] = X_test['review_density'].replace(
    [np.inf, -np.inf], review_density_median
)

print(f"Training median used for infinity replacement: {review_density_median:.4f}")
print()
print("Training distribution:")
print(X_train['review_density'].describe().round(3))

Training median used for infinity replacement: 0.2000

Training distribution:
count    39107.000
mean         3.319
std         12.897
min          0.000
25%          0.009
50%          0.200
75%          1.826
max        480.000
Name: review_density, dtype: float64


**Created Feature:**

Created `review_density` using the Ratio technique by dividing `number_of_reviews` by `availability_365`.


**Reason it was created:**

This new feature was created because using only `number_of_reviews` can be misleading. To determine the effectiveness of the number of reviews, density will rely on the availability of bookings and the number of reviews.


**Training Distribution:**
- Min: 0.000
- Median: 0.200
- Mean: 3.319
- Max: 480.0 (before capping)

In [14]:
# check outlier in new feature (review_density)
print(f"Outlier check:")
print(f"Values above 10: {(X_train['review_density'] > 10).sum()}")
print(f"Values above 50: {(X_train['review_density'] > 50).sum()}")

Outlier check:
Values above 10: 2754
Values above 50: 452


In [15]:
# Cap using training 99th percentile
cap_value = X_train['review_density'].quantile(0.99)
X_train['review_density'] = X_train['review_density'].clip(upper=cap_value)
X_test['review_density'] = X_test['review_density'].clip(upper=cap_value)

print(f"Capped at 99th percentile: {cap_value:.3f}")
print(f"New max: {X_train['review_density'].max():.3f}")

Capped at 99th percentile: 55.000
New max: 55.000


**Training Distribution:**
- Min: 0.000
- Median: 0.200
- Mean: 3.319
- New Max: 55.000 (before capping)

## Create New Features Observations


| Feature | Technique | Built From |
|---|---|---|
| `log_minimum_nights` | Log transformation | `minimum_nights` |
| `dist_times_square` | Location-based | `latitude` and `longitude` |
| `location_tier` | Binning | `dist_times_square` |
| `new_listing` | Binary flag | `number_of_reviews` |
| `review_density` | Ratio | `number_of_reviews` and `availability_365` |


# 3. Encode Categorical Features

This section converts the remaining categorical string columns to numeric. 

Ordinal Encoding: `location_tier` and `room_type`

One-Hot Encoding: `neighbourhood_group` and `neighbourhood`

In [16]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39107 entries, 0 to 39106
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   neighbourhood_group             39107 non-null  object  
 1   neighbourhood                   39107 non-null  object  
 2   latitude                        39107 non-null  float64 
 3   longitude                       39107 non-null  float64 
 4   room_type                       39107 non-null  object  
 5   minimum_nights                  39107 non-null  int64   
 6   number_of_reviews               39107 non-null  int64   
 7   reviews_per_month               39107 non-null  float64 
 8   calculated_host_listings_count  39107 non-null  float64 
 9   availability_365                39107 non-null  int64   
 10  log_minimum_nights              39107 non-null  float64 
 11  dist_to_times_square            39107 non-null  float64 
 12  location_tier     

### 3.1 Ordinal Encoding

In [18]:
# Ordinal Encoding on location_tier
from sklearn.preprocessing import OrdinalEncoder

location_tier_order = [['Very Central', 'Central', 'Outer', 'Remote']]

location_tier_encoder = OrdinalEncoder(categories=location_tier_order, 
                             handle_unknown='use_encoded_value', 
                             unknown_value=-1)

# Convert to string first to handle Categorical dtype from pd.cut
X_train['location_tier'] = X_train['location_tier'].astype(str)
X_test['location_tier'] = X_test['location_tier'].astype(str)

X_train['location_tier_encoded'] = location_tier_encoder.fit_transform(
    X_train[['location_tier']]
)
X_test['location_tier_encoded'] = location_tier_encoder.transform(
    X_test[['location_tier']]
)

# print the mapping
print('location_tier mapping:')
for i, level in enumerate(location_tier_encoder.categories_[0]):
    print(f'{level} = {i}')

location_tier mapping:
Very Central = 0
Central = 1
Outer = 2
Remote = 3


In [19]:
# Ordinal Encoding on room_type
room_type_order = [['Shared room', 'Private room', 'Entire home/apt']]

room_type_encoder = OrdinalEncoder(categories=room_type_order, 
                             handle_unknown='use_encoded_value', 
                             unknown_value=-1)

X_train['room_type_encoded'] = room_type_encoder.fit_transform(
    X_train[['room_type']]
)
X_test['room_type_encoded'] = room_type_encoder.transform(
    X_test[['room_type']]
)

# print the mapping
print("room_type mapping:")
for i, category in enumerate(room_type_encoder.categories_[0]):
    print(f"  {category} = {i}")

room_type mapping:
  Shared room = 0
  Private room = 1
  Entire home/apt = 2


In [20]:
# drop original categorical columns
X_train = X_train.drop(columns=['location_tier', 'room_type'])
X_test = X_test.drop(columns=['location_tier', 'room_type'])
print("Original ordinal columns dropped")

Original ordinal columns dropped


### 3.3 One-Hot Encoding

In [21]:
# One-Hot Encoding — neighbourhood_group
ng_dummies_train = pd.get_dummies(X_train['neighbourhood_group'], prefix='neighbourhood_group', drop_first=True, dtype=int)
X_train = pd.concat([X_train, ng_dummies_train], axis=1)
X_train = X_train.drop('neighbourhood_group', axis=1)

ng_dummies_test = pd.get_dummies(X_test['neighbourhood_group'], prefix='neighbourhood_group', drop_first=True, dtype=int)
X_test = pd.concat([X_test, ng_dummies_test], axis=1)
X_test = X_test.drop('neighbourhood_group', axis=1)

print("neighbourhood_group encoded")

neighbourhood_group encoded


In [22]:
# One-Hot Encoding — neighbourhood
nb_dummies_train = pd.get_dummies(X_train['neighbourhood'], prefix='neighbourhood', drop_first=True, dtype=int)
X_train = pd.concat([X_train, nb_dummies_train], axis=1)
X_train = X_train.drop('neighbourhood', axis=1)

nb_dummies_test = pd.get_dummies(X_test['neighbourhood'], prefix='neighbourhood', drop_first=True, dtype=int)
X_test = pd.concat([X_test, nb_dummies_test], axis=1)
X_test = X_test.drop('neighbourhood', axis=1)

print("neighbourhood encoded")

neighbourhood encoded


In [23]:
# align columns 
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"\nX_train shape after encoding: {X_train.shape}")
print(f"X_test shape after encoding:  {X_test.shape}")


X_train shape after encoding: (39107, 235)
X_test shape after encoding:  (9777, 235)


In [24]:
# check for any remaining objects
remaining_objects_train = X_train.select_dtypes(include='object').columns.tolist()
remaining_objects_test = X_test.select_dtypes(include='object').columns.tolist()

print(f"Remaining string columns in X_train: {remaining_objects_train}")
print(f"Remaining string columns in X_test:  {remaining_objects_test}")

if not remaining_objects_train and not remaining_objects_test:
    print(" All categorical columns encoded")

# print final shapes
print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

Remaining string columns in X_train: []
Remaining string columns in X_test:  []
 All categorical columns encoded

Final shapes:
  X_train: (39107, 235)
  X_test:  (9777, 235)


## Encode Categorical Features Observations

**Encoding Methods Used:**

| Column | Encoding Method | Reason |
|---|---|---|
| `location_tier` | Ordinal Encoding | Natural order exists: Very Central -> Central -> Outer -> Remote |
| `room_type` | Ordinal Encoding | Natural order in privacy and price: Shared room < Private room < Entire house/appartment |
| `neighbourhood_group` | One-Hot Encoding | 5 unordered between borough categories |
| `neighbourhood` | One-Hot Encoding | Unordered neighbourhood categories |



**Total Column Count After Encoding:**

X_train: 235 Columns

X_test:  235 Columns


The `neighbourhood` column has a lot of unique values, which is why there's a significant increase in column count.

# 4. Scale Numerical Features 

# 5. Feature Selection

# 6. Save Final Data

# Key Finding Summary
